# 실습: RAG 평가의 실제 구현

이 실습에서는 샘플 RAG 시스템을 위한 **완전한 평가 프레임워크**를 직접 구축합니다.
앞선 실습(Lab 1 Task-3)에서는 RAGAS 라이브러리가 지표를 계산해 주었다면, 이번에는 **지표를 직접 구현**하여 각 점수가 무엇을 의미하는지 확인하고, 마지막에 **Amazon Bedrock 모델 평가(RAG 평가 작업)** 와 결과를 비교합니다.

| 항목 | 내용 |
|---|---|
| **도구** | Amazon Bedrock 모델 평가, 사용자 지정 Python 평가 스크립트, 샘플 데이터세트 |
| **결과물** | 지표 대시보드가 포함된 샘플 RAG 시스템용 평가 프레임워크 |
| **적용 기술** | 검색 지표 계산, LLM-as-a-Judge 구현, 엔드 투 엔드 테스트 |
| **데이터세트** | Ground Truth 관련성 판단(qrels)을 포함한 AWS 문서 말뭉치 |

### 평가 프레임워크 구조

```
                  ┌──────────── 평가 데이터세트 (질문 · 참조 답변 · 관련성 판단 qrels) ────────────┐
                  │                                                                         │
 질문 ──▶ [검색기: BM25 / 시맨틱 / 하이브리드] ──▶ 상위 k개 문서 ──▶ [생성: Nova Lite] ──▶ 답변
                         │                                   │                          │
                         ▼                                   ▼                          ▼
              ① 검색 지표 (qrels 기반)          ② LLM-as-a-Judge: 문서 관련성     ② LLM-as-a-Judge: 충실성,
              Hit@k, Precision@k, Recall@k,       (qrels와의 일치도 = Judge 보정)     답변 관련성, 정확성
              MRR, nDCG@k, MAP
                         └──────────────────────┬──────────────────────────────────────┘
                                                ▼
                       ③ 엔드 투 엔드 테스트 (품질 게이트, 회귀 테스트, 지연 시간)
                                                ▼
                       ④ Amazon Bedrock RAG 평가 작업 (관리형 LLM-as-a-Judge) 과 교차 검증
                                                ▼
                       ⑤ 지표 대시보드 (PNG / HTML / Excel)
```

| 태스크 | 내용 |
|---|---|
| 태스크 1 | 환경 설정 |
| 태스크 2 | 평가 데이터세트 준비 (AWS 문서 말뭉치 + Ground Truth) |
| 태스크 3 | 샘플 RAG 시스템 구축 (검색기 3종 + 생성) |
| 태스크 4 | 검색 지표 계산 |
| 태스크 5 | LLM-as-a-Judge 구현 |
| 태스크 6 | 엔드 투 엔드 테스트 |
| 태스크 7 | Amazon Bedrock 모델 평가 (RAG 평가 작업) |
| 태스크 8 | 지표 대시보드 |

<i aria-hidden="true" class="fas fa-exclamation-circle" style="color:#7C5AED"></i> **주의:** **Run All Cells** 대신 셀을 하나씩 실행하세요. 태스크 6은 Bedrock을 약 150회 호출하므로 3~6분 정도 걸립니다.

## 태스크 1: 환경 설정

- 생성 모델: **amazon.nova-lite-v1:0** (앞선 실습과 동일)
- 평가(Judge) 모델: **amazon.nova-pro-v1:0** — 평가 대상보다 **더 강한 모델**을 Judge로 쓰는 것이 일반적인 모범 사례입니다. 계정에서 사용할 수 없다면 Nova Lite로 바꾸세요.
- 임베딩 모델: **amazon.titan-embed-text-v2:0** (Knowledge Base 실습과 동일)

1. 다음 셀을 실행하여 필요한 패키지를 설치합니다.

In [ ]:
%pip install -q --upgrade boto3 pandas numpy matplotlib openpyxl

2. 다음 셀을 실행하여 클라이언트와 모델 ID를 설정합니다.

In [ ]:
import json, time, re, math, random, os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import boto3
import botocore
from botocore.config import Config
import numpy as np
import pandas as pd

session = boto3.Session()
region = session.region_name or "us-east-1"

bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={"max_attempts": 3, "mode": "adaptive"})
bedrock_runtime = boto3.client("bedrock-runtime", region_name=region, config=bedrock_config)  # 모델 호출
bedrock = boto3.client("bedrock", region_name=region)                                        # 평가 작업 관리

GEN_MODEL_ID   = "amazon.nova-lite-v1:0"         # RAG 답변 생성
JUDGE_MODEL_ID = "amazon.nova-pro-v1:0"          # LLM-as-a-Judge (교차 리전 추론 프로필이 필요하면 "us.amazon.nova-pro-v1:0")
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"  # 시맨틱 검색용 임베딩

TOP_K = 3            # 생성 단계에 넘길 문서 수
OUTPUT_DIR = "rag_eval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.options.display.max_colwidth = 80
print("Region:", region)

3. 다음 셀을 실행하여 공통 헬퍼를 정의합니다. 평가는 짧은 시간에 많은 호출을 보내므로 **스로틀링 시 지수 백오프로 재시도**하고, 모든 호출의 **지연 시간과 토큰 수**를 기록합니다(엔드 투 엔드 테스트에서 사용).

In [ ]:
RETRYABLE = {"ThrottlingException", "ServiceUnavailableException", "ModelNotReadyException", "InternalServerException"}

def call_with_retry(fn, max_retries=6, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(**kwargs)
        except botocore.exceptions.ClientError as e:
            if e.response["Error"]["Code"] in RETRYABLE and attempt < max_retries - 1:
                time.sleep(min(2 ** attempt + random.random(), 30))
                continue
            raise

def converse(model_id, prompt, system=None, max_tokens=800, temperature=0.0):
    """Converse API 호출 → 텍스트, 지연 시간(ms), 토큰 수 반환"""
    kwargs = dict(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature},
    )
    if system:
        kwargs["system"] = [{"text": system}]
    t0 = time.perf_counter()
    r = call_with_retry(bedrock_runtime.converse, **kwargs)
    usage = r.get("usage", {})
    return {
        "text": r["output"]["message"]["content"][0]["text"],
        "latency_ms": (time.perf_counter() - t0) * 1000,
        "input_tokens": usage.get("inputTokens", 0),
        "output_tokens": usage.get("outputTokens", 0),
    }

def embed(text):
    body = json.dumps({"inputText": text, "dimensions": 512, "normalize": True})
    r = call_with_retry(bedrock_runtime.invoke_model, modelId=EMBED_MODEL_ID, body=body,
                        contentType="application/json", accept="application/json")
    return np.array(json.loads(r["body"].read())["embedding"], dtype=np.float32)

# 연결 확인
for m in (GEN_MODEL_ID, JUDGE_MODEL_ID):
    out = converse(m, "한 단어로 답하세요: AWS의 객체 스토리지 서비스 이름은?", max_tokens=20)
    print(f"{m:28s} → {out['text'].strip()}  ({out['latency_ms']:.0f} ms)")
print(EMBED_MODEL_ID, "→ dim", embed("hello").shape[0])

## 태스크 2: 평가 데이터세트 준비

평가 데이터세트는 세 부분으로 구성됩니다.

| 구성 요소 | 용도 | 사용하는 지표 |
|---|---|---|
| **말뭉치(corpus)** | 검색 대상 AWS 문서 | — |
| **질문 + 참조 답변(reference)** | 생성 품질의 기준 | 정확성(correctness) |
| **관련성 판단(qrels)** | 질문별로 어떤 문서가 얼마나 관련 있는지 사람이 매긴 등급<br>`2` = 핵심 근거, `1` = 부분 관련, 생략 = 무관 | Precision, Recall, MRR, nDCG |

여기서는 실습용으로 AWS 문서를 요약한 짧은 말뭉치를 사용합니다. 일부 문서는 **방해 문서(distractor)** 로 설계되어 있습니다. 예를 들어 Lambda 제한 문서와 Lambda 동시성 문서는 어휘가 겹치지만 답하는 질문이 다릅니다.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **참고:** 문서의 수치는 작성 시점 기준 요약입니다. 실무에서는 최신 AWS 문서를 원본으로 사용하세요.

4. 다음 셀을 실행하여 AWS 문서 말뭉치를 정의합니다.

In [ ]:
CORPUS = [
    {"doc_id": "s3-durability", "title": "Amazon S3 데이터 내구성과 가용성",
     "source": "https://docs.aws.amazon.com/AmazonS3/latest/userguide/DataDurability.html",
     "text": "Amazon S3 Standard 스토리지 클래스는 연간 99.999999999%(11 9s)의 객체 내구성과 99.99%의 가용성을 제공하도록 설계되었습니다. "
             "S3는 객체를 하나의 AWS 리전 안에서 최소 3개의 가용 영역(AZ)에 중복 저장하므로 하나의 AZ 전체가 손실되어도 데이터가 보존됩니다."},
    {"doc_id": "s3-storage-classes", "title": "Amazon S3 스토리지 클래스",
     "source": "https://docs.aws.amazon.com/AmazonS3/latest/userguide/storage-class-intro.html",
     "text": "S3 Standard-IA는 자주 액세스하지 않는 데이터를 위한 클래스로 최소 보관 기간 30일과 객체당 최소 과금 크기 128KB가 적용됩니다. "
             "S3 Glacier Flexible Retrieval은 최소 보관 기간이 90일이며, S3 Glacier Deep Archive는 가장 저렴한 클래스로 최소 보관 기간이 180일이고 "
             "표준 검색은 12시간 이내에 완료됩니다. 최소 보관 기간 전에 삭제하면 남은 기간만큼 요금이 부과됩니다."},
    {"doc_id": "s3-lifecycle", "title": "Amazon S3 수명 주기 관리",
     "source": "https://docs.aws.amazon.com/AmazonS3/latest/userguide/object-lifecycle-mgmt.html",
     "text": "S3 수명 주기 구성은 객체 그룹에 적용할 규칙의 집합입니다. 전환(Transition) 작업은 객체를 다른 스토리지 클래스로 옮기고, "
             "만료(Expiration) 작업은 지정한 기간이 지나면 객체를 삭제합니다. 예를 들어 생성 30일 후 Standard-IA로, 365일 후 Glacier Deep Archive로 전환하는 규칙을 만들 수 있습니다."},
    {"doc_id": "lambda-limits", "title": "AWS Lambda 할당량",
     "source": "https://docs.aws.amazon.com/lambda/latest/dg/gettingstarted-limits.html",
     "text": "Lambda 함수의 최대 실행 시간(제한 시간)은 900초(15분)입니다. 메모리는 128MB부터 10,240MB까지 1MB 단위로 설정할 수 있으며, "
             "CPU는 메모리에 비례해 할당됩니다. /tmp 임시 스토리지는 512MB에서 10,240MB까지 구성할 수 있고, 동기 호출의 요청 및 응답 페이로드는 각각 최대 6MB입니다."},
    {"doc_id": "lambda-concurrency", "title": "AWS Lambda 동시성",
     "source": "https://docs.aws.amazon.com/lambda/latest/dg/lambda-concurrency.html",
     "text": "Lambda의 리전별 기본 동시성 할당량은 계정당 1,000입니다. 예약된 동시성은 특정 함수가 사용할 수 있는 최대 동시성을 보장하고 제한합니다. "
             "프로비저닝된 동시성은 실행 환경을 미리 초기화해 두므로 콜드 스타트 지연 시간을 줄이는 데 사용합니다."},
    {"doc_id": "dynamodb-capacity", "title": "Amazon DynamoDB 읽기/쓰기 용량 모드",
     "source": "https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/capacity-mode.html",
     "text": "DynamoDB는 온디맨드 모드와 프로비저닝 모드를 제공합니다. 1 RCU(읽기 용량 단위)는 최대 4KB 항목을 초당 1회 강력한 일관성으로 읽거나 "
             "초당 2회 최종적 일관성으로 읽는 용량입니다. 1 WCU(쓰기 용량 단위)는 최대 1KB 항목을 초당 1회 쓰는 용량입니다. "
             "항목 크기는 4KB(읽기) 또는 1KB(쓰기) 단위로 올림하여 계산하며, 항목의 최대 크기는 400KB입니다."},
    {"doc_id": "dynamodb-indexes", "title": "Amazon DynamoDB 보조 인덱스",
     "source": "https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/SecondaryIndexes.html",
     "text": "글로벌 보조 인덱스(GSI)는 테이블과 다른 파티션 키를 가질 수 있고 테이블 생성 후에도 추가하거나 삭제할 수 있으며 기본 할당량은 테이블당 20개입니다. "
             "로컬 보조 인덱스(LSI)는 테이블과 같은 파티션 키를 사용하고 테이블을 생성할 때만 정의할 수 있으며 테이블당 최대 5개입니다."},
    {"doc_id": "bedrock-kb", "title": "Amazon Bedrock Knowledge Bases 개요",
     "source": "https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html",
     "text": "Amazon Bedrock Knowledge Bases는 Amazon S3 등의 데이터 소스에서 문서를 수집해 청크로 나누고 임베딩 모델로 벡터를 만들어 "
             "Amazon OpenSearch Serverless, Amazon Aurora PostgreSQL(pgvector) 등의 벡터 스토어에 저장합니다. "
             "Retrieve API는 관련 청크만 반환하고, RetrieveAndGenerate API는 검색한 청크로 파운데이션 모델의 답변까지 생성하며 출처 인용을 함께 제공합니다."},
    {"doc_id": "bedrock-kb-chunking", "title": "Amazon Bedrock Knowledge Bases 청킹 전략",
     "source": "https://docs.aws.amazon.com/bedrock/latest/userguide/kb-chunking.html",
     "text": "Knowledge Bases는 기본 청킹, 고정 크기 청킹, 계층적 청킹, 시맨틱 청킹, 청킹 없음 전략을 지원합니다. "
             "기본 청킹은 문서를 약 300토큰 크기의 청크로 나누며 문장 경계를 유지합니다. 계층적 청킹은 부모-자식 청크를 만들어 자식 청크로 검색하고 부모 청크로 더 넓은 컨텍스트를 제공합니다. "
             "시맨틱 청킹은 임베딩 유사도를 기준으로 의미가 비슷한 문장을 하나의 청크로 묶습니다."},
    {"doc_id": "bedrock-guardrails", "title": "Amazon Bedrock Guardrails",
     "source": "https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html",
     "text": "Amazon Bedrock Guardrails는 콘텐츠 필터, 거부 주제, 단어 필터, 민감한 정보(PII) 필터, 컨텍스트 근거 검사(contextual grounding check)를 제공합니다. "
             "컨텍스트 근거 검사는 모델 응답이 제공된 참조 소스에 근거하는지(grounding)와 사용자 질문과 관련 있는지(relevance)를 점수로 평가하여 RAG 응답의 환각을 탐지하고 차단합니다. "
             "ApplyGuardrail API를 사용하면 모델을 호출하지 않고도 임의의 텍스트를 가드레일로 검사할 수 있습니다."},
    {"doc_id": "bedrock-evaluation", "title": "Amazon Bedrock 모델 평가와 RAG 평가",
     "source": "https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-kb.html",
     "text": "Amazon Bedrock 평가는 프로그래밍 방식 지표, LLM-as-a-Judge 기반 자동 평가, 인간 작업자 기반 평가를 제공합니다. "
             "RAG 평가 작업은 검색 전용 평가와 검색 및 생성 평가를 지원합니다. 검색 전용 평가는 컨텍스트 관련성(context relevance)과 컨텍스트 커버리지(context coverage)를 측정하고, "
             "검색 및 생성 평가는 정확성, 완전성, 유용성, 논리적 일관성, 충실성, 인용 정밀도 등을 측정합니다. "
             "Bedrock Knowledge Bases뿐 아니라 직접 구축한 RAG 시스템의 응답을 가져와(Bring Your Own Inference) 평가할 수도 있습니다."},
    {"doc_id": "iam-policy-eval", "title": "IAM 정책 평가 로직",
     "source": "https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_policies_evaluation-logic.html",
     "text": "IAM은 기본적으로 모든 요청을 거부(암시적 거부)합니다. 자격 증명 기반 또는 리소스 기반 정책의 명시적 허용(Allow)이 있으면 요청이 허용되지만, "
             "어느 정책에든 명시적 거부(Deny)가 있으면 허용 문이 있더라도 항상 거부가 우선합니다."},
    {"doc_id": "iam-roles", "title": "IAM 역할",
     "source": "https://docs.aws.amazon.com/IAM/latest/UserGuide/id_roles.html",
     "text": "IAM 역할은 장기 자격 증명(암호나 액세스 키)이 없는 자격 증명으로, 역할을 맡으면 AWS STS가 발급하는 임시 보안 자격 증명을 받습니다. "
             "EC2 인스턴스, Lambda 함수, Amazon Bedrock 같은 AWS 서비스가 다른 리소스에 접근할 때 역할을 사용하며, 최소 권한 원칙에 따라 필요한 권한만 부여해야 합니다."},
    {"doc_id": "cloudwatch-logs", "title": "Amazon CloudWatch Logs 보존 기간",
     "source": "https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/Working-with-log-groups-and-streams.html",
     "text": "CloudWatch Logs의 로그 그룹은 기본적으로 로그를 무기한 보존(만료 없음)합니다. 보존 기간은 1일부터 10년까지 설정할 수 있으며, "
             "CloudWatch Logs Insights를 사용하면 전용 쿼리 언어로 로그를 대화형으로 검색하고 분석할 수 있습니다."},
    {"doc_id": "vpc-endpoints", "title": "VPC 엔드포인트",
     "source": "https://docs.aws.amazon.com/vpc/latest/privatelink/concepts.html",
     "text": "VPC 엔드포인트를 사용하면 인터넷 게이트웨이나 NAT 디바이스 없이 VPC에서 AWS 서비스에 비공개로 연결할 수 있습니다. "
             "게이트웨이 엔드포인트는 Amazon S3와 DynamoDB만 지원하며 추가 요금이 없습니다. 인터페이스 엔드포인트는 AWS PrivateLink를 사용하며 시간당 요금과 처리한 데이터 GB당 요금이 부과됩니다."},
    {"doc_id": "ec2-spot", "title": "Amazon EC2 스팟 인스턴스",
     "source": "https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/using-spot-instances.html",
     "text": "스팟 인스턴스는 사용되지 않는 EC2 용량을 온디맨드 요금 대비 최대 90% 할인된 가격으로 사용합니다. "
             "EC2가 용량을 회수해야 하면 인스턴스가 중단되기 2분 전에 중단 알림을 보내므로, 중단을 견딜 수 있는 배치 작업이나 상태 비저장 워크로드에 적합합니다."},
    {"doc_id": "sqs-basics", "title": "Amazon SQS 대기열",
     "source": "https://docs.aws.amazon.com/AWSSimpleQueueService/latest/SQSDeveloperGuide/sqs-queue-types.html",
     "text": "SQS는 표준 대기열과 FIFO 대기열을 제공합니다. FIFO 대기열은 메시지 순서와 정확히 한 번 처리를 보장합니다. "
             "가시성 제한 시간은 기본 30초이며 최대 12시간까지 설정할 수 있습니다. 메시지 보존 기간은 기본 4일이며 1분에서 최대 14일까지 설정할 수 있습니다."},
]
DOC = {d["doc_id"]: d for d in CORPUS}
print(f"문서 {len(CORPUS)}개")
pd.DataFrame(CORPUS)[["doc_id", "title", "source"]]

5. 다음 셀을 실행하여 평가 세트(질문, 참조 답변, 관련성 판단)를 정의합니다.

질문은 **유형(category)** 을 섞어 구성했습니다. 유형별로 지표를 나눠 보면 시스템의 약점을 찾기 쉽습니다.
- `factual` 단순 사실, `numeric` 수치 계산, `reasoning` 추론 필요, `paraphrase` 문서와 다른 표현 사용
- `out_of_scope` **말뭉치에 답이 없는 질문** → 시스템이 "모른다"고 거절해야 정답 (환각 테스트)

`must_contain`은 LLM 없이 실행되는 **결정적(deterministic) 회귀 테스트**용 키워드입니다(하나라도 포함되면 통과).

In [ ]:
EVAL_SET = [
    {"qid": "q01", "category": "factual",
     "question": "S3 Standard 스토리지 클래스의 설계상 내구성과 가용성은 각각 얼마인가요?",
     "reference": "S3 Standard는 99.999999999%(11 9s)의 내구성과 99.99%의 가용성을 제공하도록 설계되었습니다.",
     "qrels": {"s3-durability": 2, "s3-storage-classes": 1},
     "must_contain": ["99.999999999", "11 9"]},
    {"qid": "q02", "category": "factual",
     "question": "Glacier Deep Archive에 저장한 객체를 최소 며칠 동안 보관해야 추가 요금이 없나요?",
     "reference": "S3 Glacier Deep Archive의 최소 보관 기간은 180일이며, 그 전에 삭제하면 남은 기간만큼 요금이 부과됩니다.",
     "qrels": {"s3-storage-classes": 2, "s3-lifecycle": 1},
     "must_contain": ["180"]},
    {"qid": "q03", "category": "factual",
     "question": "Lambda 함수는 최대 몇 분까지 실행할 수 있고, 메모리는 최대 얼마까지 설정할 수 있나요?",
     "reference": "최대 실행 시간은 900초(15분)이고, 메모리는 최대 10,240MB까지 설정할 수 있습니다.",
     "qrels": {"lambda-limits": 2},
     "must_contain": ["15분", "900"]},
    {"qid": "q04", "category": "paraphrase",
     "question": "서버리스 함수가 처음 호출될 때 느려지는 문제를 줄이려면 어떤 기능을 써야 하나요?",
     "reference": "Lambda의 프로비저닝된 동시성을 사용하면 실행 환경을 미리 초기화해 콜드 스타트 지연 시간을 줄일 수 있습니다.",
     "qrels": {"lambda-concurrency": 2},
     "must_contain": ["프로비저닝된 동시성", "Provisioned"]},
    {"qid": "q05", "category": "numeric",
     "question": "DynamoDB에서 6KB 항목을 강력한 일관성으로 초당 1회 읽으려면 RCU가 몇 개 필요한가요?",
     "reference": "읽기 크기는 4KB 단위로 올림하므로 6KB는 8KB가 되고, 강력한 일관성 읽기 1회에 8KB/4KB = 2 RCU가 필요합니다.",
     "qrels": {"dynamodb-capacity": 2},
     "must_contain": ["2 RCU", "2개", "2RCU"]},
    {"qid": "q06", "category": "reasoning",
     "question": "이미 운영 중인 DynamoDB 테이블에 로컬 보조 인덱스를 추가할 수 있나요? 불가능하다면 대안은 무엇인가요?",
     "reference": "LSI는 테이블 생성 시에만 정의할 수 있으므로 추가할 수 없습니다. 대신 테이블 생성 후에도 추가할 수 있는 글로벌 보조 인덱스(GSI)를 사용합니다.",
     "qrels": {"dynamodb-indexes": 2},
     "must_contain": ["GSI", "글로벌 보조 인덱스"]},
    {"qid": "q07", "category": "factual",
     "question": "Bedrock Knowledge Bases의 기본 청킹은 문서를 어느 정도 크기로 나누나요?",
     "reference": "기본 청킹은 문장 경계를 유지하면서 문서를 약 300토큰 크기의 청크로 나눕니다.",
     "qrels": {"bedrock-kb-chunking": 2, "bedrock-kb": 1},
     "must_contain": ["300"]},
    {"qid": "q08", "category": "paraphrase",
     "question": "RAG 챗봇이 검색된 문서에 없는 내용을 지어내는 것을 런타임에 막으려면 Bedrock의 어떤 기능을 쓰나요?",
     "reference": "Amazon Bedrock Guardrails의 컨텍스트 근거 검사(contextual grounding check)를 사용하면 응답의 근거성과 관련성을 점수화해 환각을 탐지하고 차단할 수 있습니다.",
     "qrels": {"bedrock-guardrails": 2, "bedrock-evaluation": 1},
     "must_contain": ["근거 검사", "grounding"]},
    {"qid": "q09", "category": "factual",
     "question": "Bedrock RAG 평가 작업에서 검색 전용 평가는 어떤 지표를 측정하나요?",
     "reference": "검색 전용 평가는 컨텍스트 관련성(context relevance)과 컨텍스트 커버리지(context coverage)를 측정합니다.",
     "qrels": {"bedrock-evaluation": 2, "bedrock-kb": 1},
     "must_contain": ["컨텍스트 관련성", "context relevance"]},
    {"qid": "q10", "category": "reasoning",
     "question": "한 정책은 s3:GetObject를 허용하고 다른 정책은 같은 작업을 명시적으로 거부하면 최종 결과는 무엇인가요?",
     "reference": "명시적 거부가 항상 허용보다 우선하므로 요청은 거부됩니다.",
     "qrels": {"iam-policy-eval": 2},
     "must_contain": ["거부"]},
    {"qid": "q11", "category": "reasoning",
     "question": "프라이빗 서브넷의 EC2 인스턴스가 NAT 게이트웨이 없이, 추가 비용 없이 S3에 접근하려면 어떻게 해야 하나요?",
     "reference": "S3용 게이트웨이 VPC 엔드포인트를 사용합니다. 게이트웨이 엔드포인트는 S3와 DynamoDB를 지원하며 추가 요금이 없습니다.",
     "qrels": {"vpc-endpoints": 2},
     "must_contain": ["게이트웨이 엔드포인트", "게이트웨이 VPC 엔드포인트", "Gateway"]},
    {"qid": "q12", "category": "factual",
     "question": "SQS 메시지는 최대 며칠까지 대기열에 보존할 수 있나요?",
     "reference": "SQS 메시지 보존 기간은 기본 4일이며 최대 14일까지 설정할 수 있습니다.",
     "qrels": {"sqs-basics": 2},
     "must_contain": ["14"]},
    {"qid": "q13", "category": "out_of_scope",
     "question": "Amazon Aurora 클러스터 볼륨의 최대 스토리지 크기는 얼마인가요?",
     "reference": "제공된 문서에는 Aurora 스토리지에 대한 정보가 없으므로 답을 찾을 수 없다고 답해야 합니다.",
     "qrels": {},
     "must_contain": []},
]
print(f"평가 질문 {len(EVAL_SET)}개:", dict(Counter(q["category"] for q in EVAL_SET)))

# 무결성 검사: qrels의 doc_id가 말뭉치에 있는지
for q in EVAL_SET:
    missing = [d for d in q["qrels"] if d not in DOC]
    assert not missing, f"{q['qid']} qrels에 없는 문서: {missing}"

# 다른 도구(Bedrock 평가 등)에서 재사용할 수 있도록 JSONL로 저장
with open(f"{OUTPUT_DIR}/eval_set.jsonl", "w", encoding="utf-8") as f:
    for q in EVAL_SET:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
pd.DataFrame(EVAL_SET)[["qid", "category", "question", "qrels"]]

## 태스크 3: 샘플 RAG 시스템 구축

평가 프레임워크의 목적은 **여러 구성을 같은 기준으로 비교**하는 것입니다. 세 가지 검색기를 만들어 비교합니다.

| 구성 | 방식 | 강점 | 약점 |
|---|---|---|---|
| `bm25` | 키워드(어휘) 일치 | 정확한 용어, 숫자, 제품명 | 동의어, 다른 표현(paraphrase) |
| `semantic` | Titan 임베딩 + 코사인 유사도 | 의미가 같은 다른 표현 | 정확한 용어 일치가 약할 수 있음 |
| `hybrid` | 두 순위를 RRF(Reciprocal Rank Fusion)로 결합 | 두 방식의 장점 결합 | 계산 비용 증가 |

Bedrock Knowledge Bases의 `overrideSearchType: "HYBRID"`(Lab 1 Task-2)가 바로 세 번째 방식에 해당합니다.

6. 다음 셀을 실행하여 BM25 검색기를 구현합니다. 한국어는 조사가 붙어 공백 단위 토큰이 잘 일치하지 않으므로 **문자 bigram**을 함께 사용합니다.

In [ ]:
def tokenize(text):
    text = text.lower()
    words = re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text)                      # 영문/숫자 토큰
    hangul = re.findall(r"[가-힣]+", text)
    bigrams = [w[i:i + 2] for w in hangul for i in range(max(1, len(w) - 1))]  # 한글 문자 bigram
    return words + bigrams

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.ids = [d["doc_id"] for d in docs]
        self.tfs = [Counter(tokenize(d["title"] + " " + d["text"])) for d in docs]
        self.lens = [sum(tf.values()) for tf in self.tfs]
        self.avgdl = sum(self.lens) / len(self.lens)
        df = Counter(t for tf in self.tfs for t in tf)
        n = len(docs)
        self.idf = {t: math.log(1 + (n - c + 0.5) / (c + 0.5)) for t, c in df.items()}
        self.k1, self.b = k1, b

    def search(self, query, k):
        q = tokenize(query)
        scores = []
        for doc_id, tf, dl in zip(self.ids, self.tfs, self.lens):
            s = sum(self.idf.get(t, 0) * tf[t] * (self.k1 + 1) /
                    (tf[t] + self.k1 * (1 - self.b + self.b * dl / self.avgdl)) for t in q if t in tf)
            scores.append((doc_id, s))
        return sorted(scores, key=lambda x: -x[1])[:k]

bm25 = BM25(CORPUS)
bm25.search("Lambda 최대 실행 시간", 3)

7. 다음 셀을 실행하여 시맨틱 검색기와 하이브리드(RRF) 검색기를 구현합니다. 문서 임베딩은 한 번만 계산해 캐시합니다.

In [ ]:
doc_ids = [d["doc_id"] for d in CORPUS]
with ThreadPoolExecutor(max_workers=4) as ex:
    DOC_EMB = np.vstack(list(ex.map(lambda d: embed(d["title"] + "\n" + d["text"]), CORPUS)))
_query_emb_cache = {}

def semantic_search(query, k):
    if query not in _query_emb_cache:
        _query_emb_cache[query] = embed(query)
    sims = DOC_EMB @ _query_emb_cache[query]           # 정규화된 벡터 → 내적 = 코사인 유사도
    order = np.argsort(-sims)[:k]
    return [(doc_ids[i], float(sims[i])) for i in order]

def hybrid_search(query, k, rrf_k=60):
    """Reciprocal Rank Fusion: score = Σ 1 / (rrf_k + rank)"""
    fused = Counter()
    for results in (bm25.search(query, len(CORPUS)), semantic_search(query, len(CORPUS))):
        for rank, (doc_id, _) in enumerate(results, 1):
            fused[doc_id] += 1 / (rrf_k + rank)
    return fused.most_common(k)

RETRIEVERS = {
    "bm25": bm25.search,
    "semantic": semantic_search,
    "hybrid": hybrid_search,
}

q = "서버리스 함수가 처음 호출될 때 느려지는 문제"
for name, fn in RETRIEVERS.items():
    print(f"{name:9s}", [d for d, _ in fn(q, 3)])

8. 다음 셀을 실행하여 생성 단계를 정의합니다. 프롬프트는 **문서에만 근거**하고, **[doc_id] 형식으로 인용**하며, 답이 없으면 **정해진 거절 문장**을 쓰도록 지시합니다. 거절 문장을 고정해 두면 `out_of_scope` 질문을 코드로 채점할 수 있습니다.

In [ ]:
REFUSAL = "제공된 문서에서 답을 찾을 수 없습니다."

RAG_SYSTEM = f"""당신은 AWS 기술 문서 도우미입니다.
- <context> 안의 문서에 있는 정보만 사용해 한국어로 간결하게 답하세요.
- 사용한 문서는 문장 끝에 [doc_id] 형식으로 인용하세요.
- 문서에 답이 없으면 추측하지 말고 정확히 "{REFUSAL}" 라고만 답하세요."""

def build_context(doc_list):
    return "\n\n".join(f'<doc id="{d}">\n{DOC[d]["title"]}\n{DOC[d]["text"]}\n</doc>' for d in doc_list)

def rag_answer(question, retriever="hybrid", k=TOP_K):
    t0 = time.perf_counter()
    hits = RETRIEVERS[retriever](question, k)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    retrieved = [d for d, _ in hits]
    prompt = f"<context>\n{build_context(retrieved)}\n</context>\n\n<question>\n{question}\n</question>"
    gen = converse(GEN_MODEL_ID, prompt, system=RAG_SYSTEM, max_tokens=500)
    return {
        "retrieved": retrieved,
        "answer": gen["text"].strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen["latency_ms"],
        "input_tokens": gen["input_tokens"],
        "output_tokens": gen["output_tokens"],
    }

demo = rag_answer(EVAL_SET[4]["question"])
print("검색:", demo["retrieved"])
print("답변:", demo["answer"])

## 태스크 4: 검색 지표 계산

검색 지표는 **LLM 없이** qrels만으로 계산되므로 빠르고 저렴하며 결정적입니다. 검색 단계를 튜닝(청킹, top-k, 검색 방식, 리랭커)할 때 가장 먼저 보는 지표입니다.

| 지표 | 질문 | 계산 |
|---|---|---|
| **Hit@k** | 상위 k개 안에 관련 문서가 하나라도 있나? | 0 또는 1 |
| **Precision@k** | 상위 k개 중 관련 문서 비율은? (노이즈) | 관련 수 / k |
| **Recall@k** | 전체 관련 문서 중 몇 개를 찾았나? (누락) | 찾은 관련 수 / 전체 관련 수 |
| **MRR** | 첫 번째 관련 문서가 얼마나 위에 있나? | 1 / 첫 관련 문서 순위 |
| **nDCG@k** | **등급**(2 > 1)과 **순위**를 함께 반영한 품질 | DCG / 이상적 DCG, gain = 2^rel − 1 |
| **MAP** | 관련 문서들이 고르게 위에 있나? | 관련 문서 위치별 Precision의 평균 |

> 💡 RAG에서는 **Recall@k**가 특히 중요합니다. 검색에서 놓친 근거는 생성 단계에서 복구할 수 없기 때문입니다. 반면 Precision이 낮으면 노이즈 문서가 LLM을 혼란시키고 토큰 비용이 늘어납니다.

9. 다음 셀을 실행하여 검색 지표 함수를 구현하고 **손으로 계산할 수 있는 예제로 단위 테스트**합니다. 평가 코드 자체에 버그가 있으면 모든 결론이 틀어지므로 지표 구현도 반드시 테스트해야 합니다.

In [ ]:
def _is_rel(doc_id, qrels):
    return qrels.get(doc_id, 0) > 0

def hit_at_k(retrieved, qrels, k):
    return float(any(_is_rel(d, qrels) for d in retrieved[:k]))

def precision_at_k(retrieved, qrels, k):
    return sum(_is_rel(d, qrels) for d in retrieved[:k]) / k

def recall_at_k(retrieved, qrels, k):
    n_rel = sum(1 for v in qrels.values() if v > 0)
    return sum(_is_rel(d, qrels) for d in retrieved[:k]) / n_rel if n_rel else 0.0

def reciprocal_rank(retrieved, qrels):
    for rank, d in enumerate(retrieved, 1):
        if _is_rel(d, qrels):
            return 1 / rank
    return 0.0

def ndcg_at_k(retrieved, qrels, k):
    dcg = sum((2 ** qrels.get(d, 0) - 1) / math.log2(i + 2) for i, d in enumerate(retrieved[:k]))
    ideal = sorted(qrels.values(), reverse=True)[:k]
    idcg = sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

def average_precision(retrieved, qrels):
    n_rel = sum(1 for v in qrels.values() if v > 0)
    hits, total = 0, 0.0
    for rank, d in enumerate(retrieved, 1):
        if _is_rel(d, qrels):
            hits += 1
            total += hits / rank
    return total / n_rel if n_rel else 0.0

# ---- 단위 테스트: 관련 문서 A(2), B(1) / 검색 결과 [X, A, Y, B]
_ret, _qrels = ["X", "A", "Y", "B"], {"A": 2, "B": 1}
assert hit_at_k(_ret, _qrels, 1) == 0 and hit_at_k(_ret, _qrels, 2) == 1
assert precision_at_k(_ret, _qrels, 2) == 0.5
assert recall_at_k(_ret, _qrels, 2) == 0.5 and recall_at_k(_ret, _qrels, 4) == 1.0
assert reciprocal_rank(_ret, _qrels) == 0.5
assert abs(average_precision(_ret, _qrels) - (1/2 + 2/4) / 2) < 1e-9
assert abs(ndcg_at_k(["A", "B"], _qrels, 2) - 1.0) < 1e-9          # 이상적 순서 → 1
assert ndcg_at_k(["B", "A"], _qrels, 2) < 1.0                        # 등급 역순 → 감점
print("✅ 검색 지표 단위 테스트 통과")

10. 다음 셀을 실행하여 세 검색기를 평가합니다. 답이 없는 `out_of_scope` 질문은 관련 문서가 없으므로 검색 지표에서 제외합니다.

In [ ]:
MAX_K = 5

def evaluate_retrieval(retriever, k=TOP_K):
    rows = []
    for q in EVAL_SET:
        if not q["qrels"]:
            continue
        ret = [d for d, _ in RETRIEVERS[retriever](q["question"], MAX_K)]
        row = {"retriever": retriever, "qid": q["qid"], "category": q["category"], "retrieved": ret[:k],
               f"hit@{k}": hit_at_k(ret, q["qrels"], k),
               f"precision@{k}": precision_at_k(ret, q["qrels"], k),
               f"recall@{k}": recall_at_k(ret, q["qrels"], k),
               "mrr": reciprocal_rank(ret, q["qrels"]),
               f"ndcg@{k}": ndcg_at_k(ret, q["qrels"], k),
               "map": average_precision(ret, q["qrels"])}
        for kk in range(1, MAX_K + 1):                      # Recall@k 곡선용
            row[f"_recall@{kk}"] = recall_at_k(ret, q["qrels"], kk)
        rows.append(row)
    return rows

retrieval_df = pd.DataFrame([r for name in RETRIEVERS for r in evaluate_retrieval(name)])
RET_METRICS = [f"hit@{TOP_K}", f"precision@{TOP_K}", f"recall@{TOP_K}", "mrr", f"ndcg@{TOP_K}", "map"]

retrieval_summary = retrieval_df.groupby("retriever")[RET_METRICS].mean().reindex(list(RETRIEVERS))
retrieval_summary.style.format("{:.3f}").highlight_max(axis=0, color="#cde2fb")

11. 다음 셀을 실행하여 **질문 유형별** 그리고 **실패한 질문별**로 결과를 분석합니다. 평균만 보면 특정 유형의 실패가 가려집니다.

In [ ]:
display(retrieval_df.pivot_table(index="category", columns="retriever", values=f"recall@{TOP_K}")
        [list(RETRIEVERS)].style.format("{:.2f}"))

failed = retrieval_df[retrieval_df[f"recall@{TOP_K}"] < 1.0][["retriever", "qid", "category", "retrieved", f"recall@{TOP_K}", "mrr"]]
print("Recall@k < 1 인 질문 (근거 문서를 놓친 경우):")
display(failed)
for _, r in failed.iterrows():
    q = next(x for x in EVAL_SET if x["qid"] == r["qid"])
    print(f"[{r['retriever']}] {r['qid']} 정답 문서 {list(q['qrels'])} ↔ 검색 {r['retrieved']}")

> 💡 **확인 포인트**
> - `paraphrase` 질문(q04, q08)에서 BM25와 시맨틱 검색의 차이가 나타나나요? 문서와 다른 단어를 쓰는 질문은 어휘 기반 검색이 놓치기 쉽습니다.
> - Precision@3는 관련 문서가 1개뿐인 질문에서 최대 0.33입니다. **지표의 상한**을 이해하고 해석해야 합니다. 이런 이유로 RAG에서는 Precision보다 Recall과 MRR/nDCG를 함께 봅니다.

## 태스크 5: LLM-as-a-Judge 구현

생성된 답변은 정답 문자열과 정확히 일치하지 않으므로 BLEU/ROUGE 같은 어휘 지표로는 품질을 제대로 측정하기 어렵습니다. **LLM-as-a-Judge**는 강한 LLM에 **채점 기준(rubric)** 을 주고 점수와 이유를 받는 방식입니다.

| 지표 | 입력 | 질문 | 필요한 Ground Truth |
|---|---|---|---|
| **충실성 (faithfulness)** | 컨텍스트, 답변 | 답변의 모든 주장이 컨텍스트에 근거하는가? (환각) | 불필요 |
| **답변 관련성 (answer_relevance)** | 질문, 답변 | 질문에 직접 답하는가? | 불필요 |
| **정확성 (correctness)** | 질문, 참조 답변, 답변 | 참조 답변과 사실이 일치하는가? | 참조 답변 |
| **문서 관련성 (context_relevance)** | 질문, 문서 1개 | 이 문서가 답하는 데 필요한가? (0/1/2) | 불필요 → **qrels와 비교해 Judge를 보정** |

신뢰할 수 있는 Judge를 위한 설계 원칙:
1. **이진 또는 좁은 척도**(1~5)와 **점수별 명확한 기준**을 제공합니다.
2. **이유를 먼저 쓰고 점수를 나중에** 쓰게 하여 추론 후 판단하게 합니다.
3. **JSON 출력**으로 파싱을 안정화하고, 파싱 실패는 결측값으로 기록합니다(0점으로 처리하지 않음).
4. `temperature=0`으로 재현성을 높입니다.
5. **사람의 판단(qrels)과의 일치도**를 측정해 Judge 자체를 검증합니다.

12. 다음 셀을 실행하여 Judge 프롬프트(채점 기준)를 정의합니다.

In [ ]:
JUDGE_SYSTEM = "당신은 RAG 시스템의 출력을 평가하는 엄격하고 공정한 평가자입니다. 반드시 지정된 JSON 형식으로만 답합니다."

JUDGE_PROMPTS = {
"faithfulness": """답변이 컨텍스트에 얼마나 충실한지 평가하세요. 답변의 각 주장이 컨텍스트에서 확인되는지 검사합니다.
컨텍스트에 없는 사실(수치, 기능명 등)이 하나라도 있으면 감점합니다. 답을 찾을 수 없다고 거절한 답변은 지어낸 내용이 없으므로 5점입니다.

<context>
{context}
</context>
<answer>
{answer}
</answer>

채점 기준:
5 = 모든 주장이 컨텍스트에 근거함
4 = 사소한 표현 차이만 있고 근거 없는 사실은 없음
3 = 일부 주장이 컨텍스트에서 확인되지 않음
2 = 주요 주장이 컨텍스트에서 확인되지 않음
1 = 대부분 컨텍스트와 무관하거나 모순됨

다음 JSON 형식으로만 답하세요: {{"reason": "<근거 없는 주장 나열 또는 '없음'>", "score": <1-5>}}""",

"answer_relevance": """답변이 질문에 얼마나 직접적으로 답하는지 평가하세요. 사실 여부는 평가하지 않습니다.

<question>
{question}
</question>
<answer>
{answer}
</answer>

채점 기준:
5 = 질문의 모든 부분에 직접 답함
4 = 대부분 답하지만 불필요한 내용이 조금 있음
3 = 부분적으로만 답함
2 = 질문과 관련은 있으나 핵심에 답하지 않음
1 = 질문과 무관함
(답이 없어서 거절한 경우, 질문을 이해하고 명확히 거절했다면 3점을 주세요.)

다음 JSON 형식으로만 답하세요: {{"reason": "<한두 문장>", "score": <1-5>}}""",

"correctness": """답변을 참조 답변과 비교해 사실의 정확성을 평가하세요. 표현 방식이나 인용 형식의 차이는 무시하고 사실(수치, 기능, 결론)만 비교합니다.

<question>
{question}
</question>
<reference>
{reference}
</reference>
<answer>
{answer}
</answer>

채점 기준:
5 = 참조 답변의 핵심 사실을 모두 정확히 포함
4 = 핵심 사실은 맞지만 부가 정보가 일부 누락
3 = 일부 핵심 사실만 맞음
2 = 핵심 사실이 대부분 틀리거나 누락
1 = 참조 답변과 모순됨
(참조 답변이 '거절해야 한다'인데 답변이 거절했다면 5점, 지어낸 답을 했다면 1점)

다음 JSON 형식으로만 답하세요: {{"reason": "<한두 문장>", "score": <1-5>}}""",

"context_relevance": """다음 문서가 질문에 답하는 데 얼마나 관련 있는지 평가하세요.

<question>
{question}
</question>
<document>
{document}
</document>

채점 기준:
2 = 질문에 답하는 핵심 근거를 포함함
1 = 주제는 관련 있으나 질문의 답을 직접 포함하지 않음
0 = 관련 없음

다음 JSON 형식으로만 답하세요: {{"reason": "<한 문장>", "score": <0-2>}}""",
}
SCORE_RANGE = {"faithfulness": (1, 5), "answer_relevance": (1, 5), "correctness": (1, 5), "context_relevance": (0, 2)}

13. 다음 셀을 실행하여 Judge 함수를 구현합니다. 점수는 지표 간 비교를 위해 **0~1로 정규화**합니다.

In [ ]:
def llm_judge(metric, **fields):
    prompt = JUDGE_PROMPTS[metric].format(**fields)
    out = converse(JUDGE_MODEL_ID, prompt, system=JUDGE_SYSTEM, max_tokens=400, temperature=0.0)
    lo, hi = SCORE_RANGE[metric]
    try:
        data = json.loads(re.search(r"\{.*\}", out["text"], re.S).group(0))
        score = int(data["score"])
        if not lo <= score <= hi:
            raise ValueError(score)
        return {"score": score, "norm": (score - lo) / (hi - lo), "reason": data.get("reason", "")}
    except Exception:
        # 파싱 실패는 0점이 아니라 결측값 → 평균에서 제외하고 건수를 따로 추적
        return {"score": None, "norm": np.nan, "reason": f"PARSE_ERROR: {out['text'][:200]}"}

# 동작 확인: 일부러 환각을 넣은 답변
ctx = build_context(["lambda-limits"])
good = "Lambda 함수의 최대 실행 시간은 900초(15분)입니다. [lambda-limits]"
bad  = "Lambda 함수의 최대 실행 시간은 30분이며 GPU를 사용할 수 있습니다. [lambda-limits]"
for label, ans in [("정상 답변", good), ("환각 답변", bad)]:
    r = llm_judge("faithfulness", context=ctx, answer=ans)
    print(f"{label}: score={r['score']}  reason={r['reason']}")

14. 다음 셀을 실행하여 **Judge를 보정(calibration)** 합니다. Judge가 매긴 문서 관련성(0/1/2)을 사람이 만든 qrels와 비교합니다.

- **정확 일치율**: 등급이 정확히 같은 비율
- **이진 일치율**: 관련(≥1) / 무관(0) 판단이 같은 비율
- **Cohen's kappa**: 우연히 일치할 확률을 보정한 일치도 (0.6 이상이면 상당한 일치, 0.8 이상이면 거의 완벽한 일치)

일치도가 낮다면 Judge 점수를 그대로 믿으면 안 됩니다. 채점 기준을 고치거나 Judge 모델을 바꿔야 합니다.

In [ ]:
def cohen_kappa(a, b):
    labels = sorted(set(a) | set(b))
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(l) / n) * (b.count(l) / n) for l in labels)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

# 판정 대상: 모든 검색기의 상위 k개 문서 합집합 (질문, 문서) 쌍
pairs = sorted({(q["qid"], d) for q in EVAL_SET for name in RETRIEVERS
                for d, _ in RETRIEVERS[name](q["question"], TOP_K)})
QMAP = {q["qid"]: q for q in EVAL_SET}

def judge_pair(pair):
    qid, d = pair
    r = llm_judge("context_relevance", question=QMAP[qid]["question"],
                  document=f'{DOC[d]["title"]}\n{DOC[d]["text"]}')
    return {"qid": qid, "doc_id": d, "human": QMAP[qid]["qrels"].get(d, 0), "judge": r["score"], "reason": r["reason"]}

with ThreadPoolExecutor(max_workers=4) as ex:
    calib_df = pd.DataFrame(list(ex.map(judge_pair, pairs)))

valid = calib_df.dropna(subset=["judge"])
h, j = valid["human"].astype(int).tolist(), valid["judge"].astype(int).tolist()
calibration = {
    "pairs": len(valid),
    "exact_agreement": np.mean([x == y for x, y in zip(h, j)]),
    "binary_agreement": np.mean([(x > 0) == (y > 0) for x, y in zip(h, j)]),
    "cohen_kappa": cohen_kappa(h, j),
    "binary_kappa": cohen_kappa([int(x > 0) for x in h], [int(y > 0) for y in j]),
}
print({k: round(v, 3) if isinstance(v, float) else v for k, v in calibration.items()})
display(pd.crosstab(valid["human"], valid["judge"], rownames=["human (qrels)"], colnames=["LLM judge"]))

print("\n불일치 사례:")
display(valid[valid["human"] != valid["judge"]][["qid", "doc_id", "human", "judge", "reason"]])

> 💡 **확인 포인트**
> - 불일치는 대부분 **등급 1(부분 관련)** 경계에서 발생합니다. 사람 사이에서도 의견이 갈리는 영역입니다.
> - Judge가 체계적으로 후하거나 박하다면(예: 0을 1로 자주 판정) 그 **편향 방향**을 기록해 두고 점수 해석에 반영하세요.
> - 이 보정 결과가 좋으면, 이후 **qrels가 없는 새 질문**에도 Judge의 문서 관련성 점수를 대리 지표로 쓸 수 있습니다. Bedrock RAG 평가의 *context relevance*가 바로 이 방식입니다.

## 태스크 6: 엔드 투 엔드 테스트

이제 질문 → 검색 → 생성 → 평가 전체 파이프라인을 실행하고, 결과를 **품질 게이트(quality gate)** 로 판정합니다. 이것이 CI/CD에서 프롬프트, 모델, 청킹, 검색 설정을 바꿀 때마다 실행하는 **RAG 회귀 테스트**입니다.

질문마다 기록하는 항목:
- 검색 지표 (qrels 기반)
- LLM-as-a-Judge 점수 (충실성, 답변 관련성, 정확성)
- 결정적 검사: `must_contain` 키워드 포함 여부, `out_of_scope` 질문의 거절 여부
- 운영 지표: 검색/생성 지연 시간, 입력/출력 토큰

15. 다음 셀을 실행하여 엔드 투 엔드 평가 함수를 정의합니다.

In [ ]:
GEN_METRICS = ["faithfulness", "answer_relevance", "correctness"]

def evaluate_one(q, retriever):
    rag = rag_answer(q["question"], retriever=retriever)
    row = {"retriever": retriever, "qid": q["qid"], "category": q["category"],
           "question": q["question"], "reference": q["reference"], **rag}

    # ① 검색 지표
    if q["qrels"]:
        row.update({f"recall@{TOP_K}": recall_at_k(rag["retrieved"], q["qrels"], TOP_K),
                    "mrr": reciprocal_rank(rag["retrieved"], q["qrels"]),
                    f"ndcg@{TOP_K}": ndcg_at_k(rag["retrieved"], q["qrels"], TOP_K)})

    # ② LLM-as-a-Judge
    ctx = build_context(rag["retrieved"])
    judged = {
        "faithfulness": llm_judge("faithfulness", context=ctx, answer=rag["answer"]),
        "answer_relevance": llm_judge("answer_relevance", question=q["question"], answer=rag["answer"]),
        "correctness": llm_judge("correctness", question=q["question"], reference=q["reference"], answer=rag["answer"]),
    }
    for m, r in judged.items():
        row[m] = r["norm"]
        row[f"{m}_reason"] = r["reason"]

    # ③ 결정적 검사 (LLM 없이 재현 가능)
    refused = REFUSAL[:12] in rag["answer"]
    if q["category"] == "out_of_scope":
        row["deterministic_pass"] = refused                       # 반드시 거절해야 함
    else:
        row["deterministic_pass"] = (not refused) and any(kw.lower() in rag["answer"].lower() for kw in q["must_contain"])
    row["refused"] = refused
    row["total_ms"] = rag["retrieval_ms"] + rag["generation_ms"]
    return row

def run_e2e(retriever, max_workers=4):
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(lambda q: evaluate_one(q, retriever), EVAL_SET))

16. 다음 셀을 실행하여 세 구성 모두에 대해 엔드 투 엔드 평가를 실행합니다. (약 3~6분 소요)

In [ ]:
t0 = time.time()
e2e_rows = []
for name in RETRIEVERS:
    e2e_rows += run_e2e(name)
    print(f"✔ {name} 완료 ({time.time() - t0:.0f}s)")
e2e_df = pd.DataFrame(e2e_rows)
e2e_df.to_json(f"{OUTPUT_DIR}/e2e_results.jsonl", orient="records", lines=True, force_ascii=False)

parse_errors = e2e_df[GEN_METRICS].isna().sum().sum()
print(f"Judge 파싱 실패: {parse_errors}건")
e2e_df[["retriever", "qid", "category", "retrieved", "answer"] + GEN_METRICS + ["deterministic_pass"]]

17. 다음 셀을 실행하여 **품질 게이트**를 정의하고 구성별로 통과 여부를 판정합니다. 임계값은 서비스 요구 사항에 맞게 조정하세요. 실무에서는 이 셀의 결과를 CI 파이프라인의 통과/실패 조건으로 사용합니다.

In [ ]:
QUALITY_GATES = {                     # (지표, 방향, 임계값)
    f"recall@{TOP_K}":       (">=", 0.85),
    "mrr":                   (">=", 0.75),
    "faithfulness":          (">=", 0.85),
    "answer_relevance":      (">=", 0.75),
    "correctness":           (">=", 0.75),
    "deterministic_pass":    (">=", 0.80),
    "refusal_accuracy":      (">=", 1.00),   # out_of_scope 질문은 모두 거절해야 함
    "p95_total_ms":          ("<=", 6000),
}

def summarize(df):
    in_scope = df[df["category"] != "out_of_scope"]
    oos = df[df["category"] == "out_of_scope"]
    return {
        f"recall@{TOP_K}": in_scope[f"recall@{TOP_K}"].mean(),
        "mrr": in_scope["mrr"].mean(),
        f"ndcg@{TOP_K}": in_scope[f"ndcg@{TOP_K}"].mean(),
        "faithfulness": df["faithfulness"].mean(),
        "answer_relevance": df["answer_relevance"].mean(),
        "correctness": df["correctness"].mean(),
        "deterministic_pass": in_scope["deterministic_pass"].mean(),
        "refusal_accuracy": oos["refused"].mean() if len(oos) else np.nan,
        "false_refusal_rate": in_scope["refused"].mean(),   # 답이 있는데 거절한 비율
        "p95_total_ms": df["total_ms"].quantile(0.95),
        "avg_input_tokens": df["input_tokens"].mean(),
        "avg_output_tokens": df["output_tokens"].mean(),
    }

summary_df = pd.DataFrame({name: summarize(g) for name, g in e2e_df.groupby("retriever")}).T.reindex(list(RETRIEVERS))

def gate_pass(value, op, thr):
    return value >= thr if op == ">=" else value <= thr

gate_df = pd.DataFrame({
    name: {m: ("✅ " if gate_pass(summary_df.loc[name, m], op, thr) else "❌ ") + f"{summary_df.loc[name, m]:.2f}"
           for m, (op, thr) in QUALITY_GATES.items()}
    for name in summary_df.index
}).T
gate_df["RESULT"] = ["PASS" if all("✅" in v for v in gate_df.loc[n, list(QUALITY_GATES)]) else "FAIL" for n in gate_df.index]
print("임계값:", {m: f"{op} {thr}" for m, (op, thr) in QUALITY_GATES.items()})
gate_df

18. 다음 셀을 실행하여 **실패 사례를 진단**합니다. 검색 지표와 생성 지표를 함께 보면 문제의 위치를 찾을 수 있습니다.

| 검색 Recall | 충실성 | 정확성 | 진단 | 개선 방향 |
|---|---|---|---|---|
| 낮음 | 높음 | 낮음 | **검색 실패** — 근거가 없어 답을 못 함 | 청킹, 임베딩, 하이브리드 검색, top-k, 리랭커 |
| 높음 | 낮음 | — | **생성 환각** — 근거가 있는데 지어냄 | 프롬프트 지침, Guardrails 근거 검사, 더 강한 모델 |
| 높음 | 높음 | 낮음 | **추론/해석 실패** — 근거를 잘못 사용 | CoT 지시, 프롬프트 예시, 모델 변경 |
| 낮음 | 낮음 | 높음 | **모델 사전 지식으로 답함** — 우연히 맞음, 위험 | 문서 외 지식 사용 금지 지침 강화 |

In [ ]:
def diagnose(r):
    rec = r.get(f"recall@{TOP_K}")
    if r["category"] == "out_of_scope":
        return "OK" if r["refused"] else "환각: 범위 밖 질문에 답함"
    if pd.notna(rec) and rec < 1 and r["correctness"] < 0.75:
        return "검색 실패"
    if r["faithfulness"] < 0.75 and r["correctness"] >= 0.75:
        return "사전 지식 사용 (근거 밖)"
    if r["faithfulness"] < 0.75:
        return "생성 환각"
    if r["correctness"] < 0.75:
        return "추론/해석 실패"
    if r["refused"]:
        return "잘못된 거절"
    return "OK"

e2e_df["diagnosis"] = e2e_df.apply(diagnose, axis=1)
display(e2e_df.pivot_table(index="diagnosis", columns="retriever", values="qid", aggfunc="count", fill_value=0))

issues = e2e_df[e2e_df["diagnosis"] != "OK"]
for _, r in issues.iterrows():
    print(f"\n[{r['retriever']}] {r['qid']} ({r['category']}) → {r['diagnosis']}")
    print("  Q:", r["question"])
    print("  검색:", r["retrieved"])
    print("  A:", r["answer"][:300])
    print("  정확성 이유:", r["correctness_reason"])

19. (선택) 다음 셀은 같은 검사를 **pytest 스타일 단언(assert)** 으로 표현한 예시입니다. `tests/test_rag.py`로 옮기면 코드 변경 시 CI에서 자동 실행할 수 있습니다.

In [ ]:
BEST = summary_df["correctness"].idxmax()
best_df = e2e_df[e2e_df["retriever"] == BEST]

def test_out_of_scope_refusal():
    oos = best_df[best_df["category"] == "out_of_scope"]
    assert oos["refused"].all(), f"범위 밖 질문에 답함: {oos.loc[~oos['refused'], 'qid'].tolist()}"

def test_no_hallucination():
    bad = best_df[best_df["faithfulness"] < 0.5]
    assert bad.empty, f"충실성이 매우 낮은 답변: {bad['qid'].tolist()}"

def test_quality_gates():
    failed = [m for m, (op, thr) in QUALITY_GATES.items() if not gate_pass(summary_df.loc[BEST, m], op, thr)]
    assert not failed, f"품질 게이트 미달: {failed}"

for t in (test_out_of_scope_refusal, test_no_hallucination, test_quality_gates):
    try:
        t(); print(f"✅ {t.__name__}")
    except AssertionError as e:
        print(f"❌ {t.__name__}: {e}")
print("\n최고 구성:", BEST)

## 태스크 7: Amazon Bedrock 모델 평가 (RAG 평가 작업)

지금까지 직접 구현한 Judge를 **관리형 평가 서비스**와 교차 검증합니다. Amazon Bedrock의 RAG 평가 작업은 두 가지 방식을 지원합니다.

| 방식 | 대상 | 이 노트북에서 |
|---|---|---|
| **Bring Your Own Inference (BYOI)** | 직접 만든 RAG 시스템의 질문·검색 결과·답변을 JSONL로 제공 | 태스크 6의 최고 구성 결과를 평가 (7-A) |
| **Bedrock Knowledge Bases** | Bedrock이 KB로 검색·생성까지 직접 실행 | Lab 1의 Knowledge Base가 있으면 실행 (7-B) |

사전 준비 (콘솔 또는 IaC):
- 평가 데이터와 결과를 저장할 **S3 버킷** (Bedrock 콘솔에서 보려면 버킷에 CORS 설정 필요)
- `bedrock.amazonaws.com`이 맡을 수 있는 **IAM 서비스 역할**: S3 읽기/쓰기, Judge 모델 `bedrock:InvokeModel`, (7-B) `bedrock:Retrieve`, `bedrock:RetrieveAndGenerate` 권한

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **참고:** 평가 작업은 데이터 크기와 관계없이 보통 **15~30분 이상** 걸립니다. 작업을 시작한 뒤 태스크 8을 먼저 진행하고 나중에 돌아와 결과를 확인하세요. 버킷이나 역할이 없으면 이 태스크는 건너뛰어도 됩니다.

20. 다음 셀에 S3 버킷과 IAM 역할 ARN을 입력하고 실행합니다.

In [ ]:
S3_BUCKET     = ""   # 예: "rag-eval-123456789012"
EVAL_ROLE_ARN = ""   # 예: "arn:aws:iam::123456789012:role/BedrockEvalRole"
S3_PREFIX     = "rag-eval-lab"

BEDROCK_EVAL_READY = bool(S3_BUCKET and EVAL_ROLE_ARN)
s3 = boto3.client("s3", region_name=region)
print("Bedrock 평가 작업 실행:", "가능" if BEDROCK_EVAL_READY else "건너뜀 (S3_BUCKET / EVAL_ROLE_ARN 미설정)")

### 7-A. 직접 구축한 RAG 시스템 평가 (BYOI)

21. 다음 셀을 실행하여 태스크 6의 결과를 Bedrock RAG 평가 데이터 형식(JSONL)으로 변환합니다. 한 줄이 한 대화이며 `prompt`(질문), `referenceResponses`(참조 답변), `output`(우리 시스템의 답변과 검색 결과)을 포함합니다. `knowledgeBaseIdentifier`는 작업 생성 시의 `ragSourceIdentifier`와 같아야 합니다.

In [ ]:
RAG_SOURCE_ID = f"custom-{BEST}-rag"

def to_bedrock_record(r):
    return {"conversationTurns": [{
        "prompt": {"content": [{"text": r["question"]}]},
        "referenceResponses": [{"content": [{"text": r["reference"]}]}],
        "output": {
            "text": r["answer"],
            "modelIdentifier": GEN_MODEL_ID,
            "knowledgeBaseIdentifier": RAG_SOURCE_ID,
            "retrievedPassages": {"retrievalResults": [
                {"name": d, "content": {"text": DOC[d]["text"]}} for d in r["retrieved"]
            ]},
        },
    }]}

byoi_path = f"{OUTPUT_DIR}/bedrock_byoi_dataset.jsonl"
with open(byoi_path, "w", encoding="utf-8") as f:
    for _, r in best_df.iterrows():
        f.write(json.dumps(to_bedrock_record(r), ensure_ascii=False) + "\n")
print(open(byoi_path, encoding="utf-8").readline()[:600], "...")

22. 다음 셀을 실행하여 평가 작업을 생성합니다. Bedrock의 내장 지표(`Builtin.*`)를 사용하며, Judge 모델은 직접 구현한 Judge와 같은 모델로 지정해 비교 조건을 맞춥니다.

| Bedrock 지표 | 직접 구현한 지표 |
|---|---|
| `Builtin.Correctness` | correctness |
| `Builtin.Faithfulness` | faithfulness |
| `Builtin.Helpfulness` | answer_relevance (유사) |
| `Builtin.Completeness`, `Builtin.LogicalCoherence` | (추가 지표) |

In [ ]:
eval_job_arn = None
if BEDROCK_EVAL_READY:
    dataset_key = f"{S3_PREFIX}/input/bedrock_byoi_dataset.jsonl"
    s3.upload_file(byoi_path, S3_BUCKET, dataset_key)
    job_name = f"rag-eval-byoi-{time.strftime('%Y%m%d-%H%M%S')}"

    resp = bedrock.create_evaluation_job(
        jobName=job_name,
        roleArn=EVAL_ROLE_ARN,
        applicationType="RagEvaluation",
        evaluationConfig={"automated": {
            "datasetMetricConfigs": [{
                "taskType": "QuestionAndAnswer",
                "dataset": {"name": "aws-docs-rag-eval", "datasetLocation": {"s3Uri": f"s3://{S3_BUCKET}/{dataset_key}"}},
                "metricNames": ["Builtin.Correctness", "Builtin.Completeness", "Builtin.Helpfulness",
                                "Builtin.LogicalCoherence", "Builtin.Faithfulness"],
            }],
            "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": JUDGE_MODEL_ID}]},
        }},
        inferenceConfig={"ragConfigs": [{
            "precomputedRagSourceConfig": {"retrieveAndGenerateSourceConfig": {"ragSourceIdentifier": RAG_SOURCE_ID}}
        }]},
        outputDataConfig={"s3Uri": f"s3://{S3_BUCKET}/{S3_PREFIX}/output/"},
    )
    eval_job_arn = resp["jobArn"]
    print("평가 작업 생성:", eval_job_arn)
else:
    print("건너뜀")

### 7-B. (선택) Bedrock Knowledge Base 평가

Lab 1에서 만든 Knowledge Base(AnyCompany Financial 10-K)가 있다면, Bedrock이 **검색과 생성까지 직접 실행**하며 평가합니다. 이 경우 데이터세트에는 `output` 없이 질문과 참조 답변만 넣습니다. 말뭉치가 다르므로 질문은 Lab 1 Task-3의 질문을 사용합니다.

23. Knowledge Base가 있는 경우에만 다음 셀을 실행합니다.

In [ ]:
kb_eval_job_arn = None
kb_id = None
try:
    kbs = boto3.client("bedrock-agent", region_name=region).list_knowledge_bases(maxResults=1)["knowledgeBaseSummaries"]
    kb_id = kbs[0]["knowledgeBaseId"] if kbs else None
except botocore.exceptions.ClientError as e:
    print("Knowledge Base 조회 실패:", e.response["Error"]["Code"])

if BEDROCK_EVAL_READY and kb_id:
    kb_questions = [   # Lab 1 Task-3의 질문/정답
        ("What was the primary reason for the increase in net cash by operating activities for AnyCompany Financial in 2021?",
         "An increase in net cash provided by operating activities was primarily due to increases in net income and favorable changes in operating assets and liabilities."),
        ("What was the primary source of cash inflows from financing activities for AnyCompany Financial in 2021?",
         "The primary source of cash inflows from financing activities for AnyCompany Financial in 2021 was an increase in proceeds from the issuance of common stock and long-term debt."),
    ]
    kb_path = f"{OUTPUT_DIR}/bedrock_kb_dataset.jsonl"
    with open(kb_path, "w", encoding="utf-8") as f:
        for qtext, ref in kb_questions:
            f.write(json.dumps({"conversationTurns": [{
                "prompt": {"content": [{"text": qtext}]},
                "referenceResponses": [{"content": [{"text": ref}]}],
            }]}) + "\n")
    kb_key = f"{S3_PREFIX}/input/bedrock_kb_dataset.jsonl"
    s3.upload_file(kb_path, S3_BUCKET, kb_key)

    resp = bedrock.create_evaluation_job(
        jobName=f"rag-eval-kb-{time.strftime('%Y%m%d-%H%M%S')}",
        roleArn=EVAL_ROLE_ARN,
        applicationType="RagEvaluation",
        evaluationConfig={"automated": {
            "datasetMetricConfigs": [{
                "taskType": "QuestionAndAnswer",
                "dataset": {"name": "anycompany-10k", "datasetLocation": {"s3Uri": f"s3://{S3_BUCKET}/{kb_key}"}},
                "metricNames": ["Builtin.Correctness", "Builtin.Completeness", "Builtin.Faithfulness",
                                "Builtin.CitationPrecision", "Builtin.CitationCoverage"],
            }],
            "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": JUDGE_MODEL_ID}]},
        }},
        inferenceConfig={"ragConfigs": [{"knowledgeBaseConfig": {"retrieveAndGenerateConfig": {
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": kb_id,
                "modelArn": GEN_MODEL_ID,
                "retrievalConfiguration": {"vectorSearchConfiguration": {"numberOfResults": 5}},
            },
        }}}]},
        outputDataConfig={"s3Uri": f"s3://{S3_BUCKET}/{S3_PREFIX}/output-kb/"},
    )
    kb_eval_job_arn = resp["jobArn"]
    print("KB 평가 작업 생성:", kb_eval_job_arn)
else:
    print("건너뜀 (Knowledge Base 또는 S3/역할 미설정)")

24. 다음 셀을 실행하여 작업 상태를 확인합니다. `Completed`가 될 때까지 몇 분 간격으로 다시 실행하세요.

In [ ]:
for arn in filter(None, [eval_job_arn, kb_eval_job_arn]):
    job = bedrock.get_evaluation_job(jobIdentifier=arn)
    print(f"{job['jobName']}: {job['status']}", job.get("failureMessages", ""))

25. 작업이 `Completed`가 되면 다음 셀을 실행하여 결과를 읽고 **직접 구현한 Judge와 비교**합니다. 결과 JSONL에는 질문별로 지표 점수(`results`)와 Judge의 설명이 들어 있습니다.

In [ ]:
def load_bedrock_results(output_prefix):
    """출력 경로 아래의 모든 JSONL을 읽어 (질문, 지표, 점수) 행으로 변환"""
    rows = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=output_prefix):
        for obj in page.get("Contents", []):
            if not obj["Key"].endswith(".jsonl"):
                continue
            body = s3.get_object(Bucket=S3_BUCKET, Key=obj["Key"])["Body"].read().decode("utf-8")
            for line in body.splitlines():
                rec = json.loads(line)
                for turn in rec.get("conversationTurns", []):
                    question = turn["prompt"]["content"][0]["text"]
                    for res in turn.get("results", []):
                        rows.append({"question": question, "metric": res.get("metricName"),
                                     "score": res.get("result"), "explanation": res.get("evaluatorDetails", "")})
    return pd.DataFrame(rows)

bedrock_eval_df = pd.DataFrame()
if eval_job_arn and bedrock.get_evaluation_job(jobIdentifier=eval_job_arn)["status"] == "Completed":
    bedrock_eval_df = load_bedrock_results(f"{S3_PREFIX}/output/")
    wide = bedrock_eval_df.pivot_table(index="question", columns="metric", values="score")
    mine = best_df.set_index("question")[GEN_METRICS]
    compare = wide.join(mine, how="inner")
    display(compare.round(2))

    pairs_to_compare = [("Builtin.Correctness", "correctness"), ("Builtin.Faithfulness", "faithfulness"),
                        ("Builtin.Helpfulness", "answer_relevance")]
    print("직접 구현한 Judge ↔ Bedrock 평가 (평균 / 상관계수)")
    for b, m in pairs_to_compare:
        if b in compare:
            print(f"  {m:17s} 직접={compare[m].mean():.2f}  Bedrock={compare[b].mean():.2f}  r={compare[m].corr(compare[b]):.2f}")
else:
    print("아직 결과가 없습니다. (작업 미생성 또는 진행 중)")

> 💡 **확인 포인트**
> - 두 평가의 **평균 수준**보다 **질문 간 순위가 비슷한지**(상관계수)가 더 중요합니다. 점수 척도와 채점 기준이 다르면 절대값은 달라질 수 있습니다.
> - 크게 엇갈리는 질문은 직접 읽어 보고 어느 Judge가 맞는지 판단하세요. 그 결과로 채점 기준을 개선합니다.
> - 관리형 평가는 **표준화된 지표, 콘솔 보고서, 작업 이력**을 제공하고, 사용자 지정 스크립트는 **빠른 반복과 도메인 특화 지표**(예: 거절 정확도, 키워드 회귀 테스트)에 유리합니다. 실무에서는 두 가지를 함께 사용합니다.

## 태스크 8: 지표 대시보드

26. 다음 셀을 실행하여 평가 결과를 하나의 대시보드로 시각화합니다.

| 패널 | 보여 주는 것 |
|---|---|
| ① 검색 지표 비교 | 구성별 Hit/Precision/Recall/MRR/nDCG |
| ② Recall@k 곡선 | k를 늘리면 근거를 얼마나 더 찾는가 → top-k 선택 근거 |
| ③ 생성 품질 (LLM-as-a-Judge) | 구성별 충실성, 답변 관련성, 정확성 |
| ④ 질문별 히트맵 | 최고 구성에서 어떤 질문이 약한가 |
| ⑤ Judge 보정 | 사람(qrels)과 LLM Judge의 문서 관련성 판단 일치 |
| ⑥ 품질 대 지연 시간 | 정확성과 p95 지연 시간의 트레이드오프 |

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"bm25": "#2a78d6", "semantic": "#eb6834", "hybrid": "#1baf7a"}   # 구성별 고정 색상
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlelocation": "left",
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

def grouped_bars(ax, table, metrics, title):
    names = list(table.index)
    width = 0.8 / len(names)
    x = np.arange(len(metrics))
    for i, n in enumerate(names):
        ax.bar(x + (i - (len(names) - 1) / 2) * width, table.loc[n, metrics], width * 0.9,
               color=COLORS[n], label=n, edgecolor="white", linewidth=1)
    ax.set_xticks(x, metrics)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_title(title)
    ax.legend(frameon=False, ncol=len(names), loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8)

fig, axes = plt.subplots(2, 3, figsize=(18, 10.5))
fig.suptitle(f"RAG Evaluation Dashboard  ·  {len(EVAL_SET)} questions · gen={GEN_MODEL_ID} · judge={JUDGE_MODEL_ID}",
             x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)

# ① 검색 지표
grouped_bars(axes[0, 0], retrieval_summary, RET_METRICS, "① Retrieval metrics (qrels)\n")

# ② Recall@k 곡선
ax = axes[0, 1]
ks = list(range(1, MAX_K + 1))
for n in RETRIEVERS:
    ys = [retrieval_df[retrieval_df["retriever"] == n][f"_recall@{k}"].mean() for k in ks]
    ax.plot(ks, ys, color=COLORS[n], linewidth=2, marker="o", markersize=6, label=n)
    ax.annotate(n, (ks[-1], ys[-1]), xytext=(6, 0), textcoords="offset points", color=MUTED, va="center", fontsize=8)
ax.axvline(TOP_K, color=MUTED, linestyle=":", linewidth=1)
ax.text(TOP_K, 0.02, f" TOP_K={TOP_K}", color=MUTED, fontsize=8)
ax.set_xticks(ks); ax.set_xlim(0.8, MAX_K + 0.8); ax.set_ylim(0, 1.05)
ax.set_xlabel("k"); ax.grid(axis="y", color=GRID, linewidth=0.6)
ax.set_title("② Recall@k curve\n"); ax.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8)

# ③ 생성 품질
grouped_bars(axes[0, 2], summary_df, GEN_METRICS + ["deterministic_pass"], "③ Generation quality (LLM-as-a-Judge)\n")

# ④ 질문별 히트맵 (최고 구성)
ax = axes[1, 0]
heat_cols = [f"recall@{TOP_K}", "mrr", "faithfulness", "answer_relevance", "correctness"]
heat = best_df.set_index("qid")[heat_cols].astype(float)
im = ax.imshow(heat.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heat_cols)), heat_cols, rotation=20, ha="right")
ax.set_yticks(range(len(heat)), [f"{qid} · {QMAP[qid]['category']}" for qid in heat.index])
ax.spines[:].set_visible(False)
for (i, jx), v in np.ndenumerate(heat.values):
    if np.isnan(v):
        ax.text(jx, i, "n/a", ha="center", va="center", fontsize=7, color=MUTED)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
ax.set_title(f"④ Per-question scores ({BEST})")

# ⑤ Judge 보정
ax = axes[1, 1]
cm = pd.crosstab(valid["human"], valid["judge"]).reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0)
ax.imshow(cm.values, cmap="Blues")
for (i, jx), v in np.ndenumerate(cm.values):
    ax.text(jx, i, int(v), ha="center", va="center", fontsize=11,
            color="white" if v > cm.values.max() * 0.6 else INK)
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xlabel("LLM judge grade"); ax.set_ylabel("Human qrels grade")
ax.spines[:].set_visible(False)
ax.set_title(f"⑤ Judge calibration  ·  κ={calibration['cohen_kappa']:.2f}, "
             f"binary agreement={calibration['binary_agreement']:.0%}")

# ⑥ 품질 대 지연 시간
ax = axes[1, 2]
for n in summary_df.index:
    ax.scatter(summary_df.loc[n, "p95_total_ms"], summary_df.loc[n, "correctness"], s=120, color=COLORS[n],
               edgecolor="white", linewidth=2, zorder=3)
    ax.annotate(f"{n}  ({gate_df.loc[n, 'RESULT']})", (summary_df.loc[n, "p95_total_ms"], summary_df.loc[n, "correctness"]),
                xytext=(8, 4), textcoords="offset points", color=INK, fontsize=9)
ax.axhline(QUALITY_GATES["correctness"][1], color=MUTED, linestyle=":", linewidth=1)
ax.axvline(QUALITY_GATES["p95_total_ms"][1], color=MUTED, linestyle=":", linewidth=1)
ax.set_xlabel("p95 end-to-end latency (ms)"); ax.set_ylabel("correctness (0-1)")
ax.set_ylim(0, 1.05); ax.grid(color=GRID, linewidth=0.6); ax.set_axisbelow(True)
ax.set_title("⑥ Quality vs latency (dotted = gate)")

fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(f"{OUTPUT_DIR}/dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

27. 다음 셀을 실행하여 대시보드의 **표 보기**와 함께 결과를 HTML 보고서와 Excel 파일로 내보냅니다. 차트만으로는 정확한 값을 읽기 어려우므로 표를 함께 제공합니다.

In [ ]:
display(summary_df.style.format("{:.3f}").background_gradient(
    cmap="Blues", subset=[f"recall@{TOP_K}", "mrr", f"ndcg@{TOP_K}"] + GEN_METRICS, vmin=0, vmax=1))

# Excel: 시트별 원본 데이터
with pd.ExcelWriter(f"{OUTPUT_DIR}/rag_eval_results.xlsx", engine="openpyxl") as xw:
    summary_df.to_excel(xw, sheet_name="summary")
    gate_df.to_excel(xw, sheet_name="quality_gates")
    retrieval_df.drop(columns=[c for c in retrieval_df if c.startswith("_")]).to_excel(xw, sheet_name="retrieval", index=False)
    e2e_df.to_excel(xw, sheet_name="e2e", index=False)
    calib_df.to_excel(xw, sheet_name="judge_calibration", index=False)
    if not bedrock_eval_df.empty:
        bedrock_eval_df.to_excel(xw, sheet_name="bedrock_eval", index=False)

# HTML 보고서: 대시보드 이미지 + 표
import base64
img = base64.b64encode(open(f"{OUTPUT_DIR}/dashboard.png", "rb").read()).decode()
html = f"""<html><head><meta charset="utf-8"><title>RAG Evaluation Report</title>
<style>body{{font-family:sans-serif;margin:24px;color:#0b0b0b;background:#fcfcfb}} table{{border-collapse:collapse;font-size:13px}}
td,th{{border:1px solid #e4e3df;padding:4px 8px}} th{{background:#f0efec}}</style></head><body>
<h1>RAG Evaluation Report</h1><p>{time.strftime('%Y-%m-%d %H:%M')} · 생성 {GEN_MODEL_ID} · Judge {JUDGE_MODEL_ID} · 최고 구성 <b>{BEST}</b></p>
<img src="data:image/png;base64,{img}" style="max-width:100%">
<h2>품질 게이트</h2>{gate_df.to_html()}
<h2>요약 지표</h2>{summary_df.round(3).to_html()}
<h2>Judge 보정</h2>{pd.Series(calibration).round(3).to_frame('value').to_html()}
<h2>질문별 결과 ({BEST})</h2>{best_df[['qid','category','question','retrieved','answer'] + GEN_METRICS + ['diagnosis']].round(2).to_html(index=False)}
</body></html>"""
open(f"{OUTPUT_DIR}/rag_eval_report.html", "w", encoding="utf-8").write(html)
print("저장 완료:", sorted(os.listdir(OUTPUT_DIR)))

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **참고:** 왼쪽 탐색 창의 **rag_eval_outputs** 폴더에서 `dashboard.png`, `rag_eval_report.html`, `rag_eval_results.xlsx`를 확인할 수 있습니다.

---
# 📝 학습 정리

## 1. 실습 흐름과 배운 점

| 단계 | 한 일 | 배운 점 |
|---|---|---|
| 태스크 2 | 말뭉치, 참조 답변, 등급별 관련성 판단(qrels) 구성 | 평가 품질은 **데이터세트 품질**이 결정한다. 범위 밖 질문과 다른 표현 질문을 반드시 포함 |
| 태스크 3 | BM25 / 시맨틱 / 하이브리드 검색기 | 평가 프레임워크는 **구성 비교**를 위해 존재한다 |
| 태스크 4 | Hit, Precision, Recall, MRR, nDCG, MAP 직접 구현 + 단위 테스트 | 검색 지표는 빠르고 결정적이다. RAG에서는 **Recall@k**가 상한을 결정한다 |
| 태스크 5 | 채점 기준이 있는 LLM-as-a-Judge + qrels로 보정 | Judge도 **검증 대상**이다. 사람 판단과의 일치도(kappa)를 측정한다 |
| 태스크 6 | 엔드 투 엔드 실행, 품질 게이트, 실패 진단, pytest 스타일 테스트 | 검색 지표 + 생성 지표를 **함께** 보아야 실패 위치를 찾을 수 있다 |
| 태스크 7 | Bedrock RAG 평가 작업 (BYOI / KB) | 관리형 평가로 **표준화와 교차 검증**, 사용자 지정 스크립트로 **도메인 특화와 빠른 반복** |
| 태스크 8 | 대시보드와 보고서 | 품질, 비용, 지연 시간의 **트레이드오프**를 한눈에 보여 준다 |

## 2. 핵심 메시지

1. **측정할 수 없으면 개선할 수 없다.** 검색과 생성을 분리해 측정해야 어느 단계를 고칠지 알 수 있습니다.
2. **평가 도구 자체를 검증하라.** 지표 코드는 단위 테스트로, LLM Judge는 사람 판단과의 일치도로 검증합니다.
3. **평가는 일회성이 아니라 회귀 테스트다.** 프롬프트, 모델, 청킹, 검색 설정을 바꿀 때마다 같은 데이터세트와 품질 게이트로 다시 실행합니다.

## 3. 직접 해 보기

- `TOP_K`를 5로 바꾸고 Recall, Precision, 토큰 수, 정확성이 어떻게 변하는지 확인하세요.
- 말뭉치에 질문과 비슷하지만 틀린 정보를 담은 **방해 문서**를 추가하고 충실성과 정확성의 변화를 관찰하세요.
- `JUDGE_MODEL_ID`를 Nova Lite로 바꿔 Judge 보정 결과(kappa)가 어떻게 달라지는지 비교하세요.
- `RAG_SYSTEM` 프롬프트에서 인용 지침을 제거하고 충실성 점수의 변화를 확인하세요.